In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pyliftover import LiftOver

# Chain file downloaded from:
# https://hgdownload.soe.ucsc.edu/goldenPath/mm10/liftOver/
chain_file = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/liftover/mm10ToMm39.over.chain.gz"
lo = LiftOver(chain_file)

In [3]:
# Load in the RH and DT ATSE files 
# The EasySci ATSEs are in mm9...

# Load in ATSEs from Tabula Muris Senis (mm10) 
TMS_atses = ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSEmap/output/ATSEfiles/TMS_atse_file_unanno_also_2025-01-30_19-24-18.txt.gz"

TMS_atses = pd.read_csv(TMS_atses, sep="\t")

In [4]:
TMS_atses[["chrom", "start", "end", "strand"]] = TMS_atses.junction_id.str.split("_", expand=True)
TMS_atses['chrom'] = TMS_atses['chrom'].astype(str)
TMS_atses['start'] = TMS_atses['start'].astype(int)
TMS_atses['end'] = TMS_atses['end'].astype(int)

In [5]:
# Function to convert a single row of coordinates
def liftover_row(row):
    chrom, start, end = row['chrom'], row['start'], row['end']
    # Lift over start and end coordinates
    new_start = lo.convert_coordinate(chrom, start)
    new_end = lo.convert_coordinate(chrom, end)

    # Check if both start and end were successfully converted
    if new_start and new_end:
        # Extract the new chromosome and positions
        new_chrom = new_start[0][0]
        new_start_pos = int(new_start[0][1])
        new_end_pos = int(new_end[0][1])
        return pd.Series([new_chrom, new_start_pos, new_end_pos])
    else:
        # Return None for unmapped coordinates
        return pd.Series([None, None, None])

# Apply the liftover function to each row
TMS_atses[['new_chrom', 'new_start', 'new_end']] = TMS_atses.apply(liftover_row, axis=1)

# Create the new_junction_id column
TMS_atses['new_junction_id'] = TMS_atses.apply(
    lambda row: (
        f"{row['new_chrom']}_{int(row['new_start'])}_{int(row['new_end'])}_{row['strand']}"
        if pd.notnull(row['new_chrom']) and pd.notnull(row['new_start']) and pd.notnull(row['new_end'])
        else None
    ),
    axis=1
)

In [11]:
TMS_atses.iloc[1]

event_id                                ATSE_0
gene_id                  ENSMUSG00000033845.13
gene_name                               Mrpl15
num_junctions                                4
event_type                             complex
annotation_status                         both
junction_id             chr1_4776801_4777524_-
chrom                                     chr1
start                                  4776801
end                                    4777524
strand                                       -
cells                                    21515
total_score                            1230743
splice_motif                             GT-AG
donor_seq                                   GT
acceptor_seq                                AG
label_5_prime                  annotated on 5'
label_3_prime                  annotated on 3'
position_off_5_prime                       1.0
position_off_3_prime                       0.0
new_chrom                                 chr1
new_start    

In [7]:
# save junction_id and new_junction_id to a file
TMS_atses[['junction_id', 'new_junction_id']].to_csv("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/EasySci2024/LeafletFA/ATSEmap/TMS_EASYSCI_junction_id_mapping.txt", sep="\t", index=False)

In [ ]:
# lost code but took TMS atse, did liftover and saved new version for easysci analysis 
#s aved it ehre /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/EasySci2024/LeafletFA/ATSEmap/EASYSCI_from_TMS_liftover_atse_file_unanno_also_2025-02-23_12-29-23.txt.gz